<a href="https://colab.research.google.com/github/Linda-Masia/MIT-805---Vincent-Mabuza-Linda-Masia/blob/Some-805/Some_805%20v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import os
import requests

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"

def get_remote_file_size(url):
    """Get file size via HTTP HEAD request without downloading."""
    resp = requests.head(url, allow_redirects=True)
    return int(resp.headers.get("content-length", 0))

def select_files_for_target_size(years, target_gb=12):
    """Select monthly files cumulatively until target size (in GB) is reached."""
    target_bytes = target_gb * (1024 ** 3)
    selected = []
    running_total = 0

    for year in years:
        for month in range(1, 13):
            url = BASE_URL.format(year=year, month=month)
            try:
                size = get_remote_file_size(url)
            except Exception as e:
                print(f"Skipping {year}-{month:02d}: {e}")
                continue

            if size == 0:
                continue  # file doesn't exist for that month/year

            selected.append((year, month, url, size))
            running_total += size
            print(f"{year}-{month:02d}: {size / (1024**3):.2f} GB (running total: {running_total / (1024**3):.2f} GB)")

            if running_total >= target_bytes:
                return selected, running_total

    return selected, running_total

# Search across 2013-2023 until we hit ~12GB
selected_files, total_size = select_files_for_target_size(years=[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023], target_gb=12)
print(f"\nSelected {len(selected_files)} files, total {total_size / (1024**3):.2f} GB")

2013-01: 0.17 GB (running total: 0.17 GB)
2013-02: 0.16 GB (running total: 0.33 GB)
2013-03: 0.18 GB (running total: 0.51 GB)
2013-04: 0.17 GB (running total: 0.68 GB)
2013-05: 0.18 GB (running total: 0.86 GB)
2013-06: 0.17 GB (running total: 1.03 GB)
2013-07: 0.16 GB (running total: 1.19 GB)
2013-08: 0.14 GB (running total: 1.33 GB)
2013-09: 0.16 GB (running total: 1.49 GB)
2013-10: 0.18 GB (running total: 1.67 GB)
2013-11: 0.17 GB (running total: 1.83 GB)
2013-12: 0.16 GB (running total: 2.00 GB)
2014-01: 0.16 GB (running total: 2.16 GB)
2014-02: 0.15 GB (running total: 2.31 GB)
2014-03: 0.18 GB (running total: 2.49 GB)
2014-04: 0.17 GB (running total: 2.66 GB)
2014-05: 0.17 GB (running total: 2.83 GB)
2014-06: 0.16 GB (running total: 2.99 GB)
2014-07: 0.15 GB (running total: 3.15 GB)
2014-08: 0.16 GB (running total: 3.31 GB)
2014-09: 0.17 GB (running total: 3.48 GB)
2014-10: 0.18 GB (running total: 3.66 GB)
2014-11: 0.17 GB (running total: 3.83 GB)
2014-12: 0.17 GB (running total: 4

In [27]:
import os
import requests

DOWNLOAD_DIR = "data/working"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

for year, month, url, size in selected_files:
    filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
    filepath = os.path.join(DOWNLOAD_DIR, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename} ({size / (1024**3):.2f} GB)...")
        resp = requests.get(url, stream=True)
        with open(filepath, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
    else:
        print(f"Skipping {filename}: already exists.")

Skipping yellow_tripdata_2013-01.parquet: already exists.
Skipping yellow_tripdata_2013-02.parquet: already exists.
Skipping yellow_tripdata_2013-03.parquet: already exists.
Skipping yellow_tripdata_2013-04.parquet: already exists.
Skipping yellow_tripdata_2013-05.parquet: already exists.
Skipping yellow_tripdata_2013-06.parquet: already exists.
Skipping yellow_tripdata_2013-07.parquet: already exists.
Skipping yellow_tripdata_2013-08.parquet: already exists.
Skipping yellow_tripdata_2013-09.parquet: already exists.
Skipping yellow_tripdata_2013-10.parquet: already exists.
Skipping yellow_tripdata_2013-11.parquet: already exists.
Skipping yellow_tripdata_2013-12.parquet: already exists.
Skipping yellow_tripdata_2014-01.parquet: already exists.
Skipping yellow_tripdata_2014-02.parquet: already exists.
Skipping yellow_tripdata_2014-03.parquet: already exists.
Skipping yellow_tripdata_2014-04.parquet: already exists.
Skipping yellow_tripdata_2014-05.parquet: already exists.
Skipping yello

In [28]:
import os
import pandas as pd


def get_dir_size_gb(path):
    """Total size in GB of all files under a directory (recursively)."""
    total_bytes = 0
    if not os.path.exists(path):
        return 0.0 # Return 0 if path does not exist
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_bytes += os.path.getsize(fp)
    return total_bytes / (1024 ** 3)


def list_files_with_sizes(path):
    """Return a DataFrame of individual files and their sizes (MB) in a directory."""
    rows = []
    if not os.path.exists(path):
        # Return an empty DataFrame with expected columns if the path doesn't exist
        return pd.DataFrame(columns=["file", "size_mb"])

    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            size_mb = os.path.getsize(fp) / (1024 ** 2)
            rows.append({"file": f, "size_mb": round(size_mb, 2)})

    df = pd.DataFrame(rows)
    # Check if the DataFrame is empty before sorting
    if not df.empty:
        return df.sort_values("file").reset_index(drop=True)
    else:
        # Return an empty DataFrame with expected columns if no files were found
        return pd.DataFrame(columns=["file", "size_mb"])


if __name__ == "__main__":
    RAW_DIR = "data/raw"
    WORKING_DIR = "data/working"

    print("=" * 50)
    print("DATASET SIZE REPORT (Raw & Working)")
    print("=" * 50)

    for label, path in [("Raw", RAW_DIR), ("Working", WORKING_DIR)]:
        if os.path.exists(path):
            size_gb = get_dir_size_gb(path)
            n_files = sum(len(files) for _, _, files in os.walk(path))
            print(f"{label:<12} | {size_gb:>8.2f} GB | {n_files} file(s) | {path}")
        else:
            print(f"{label:<12} | NOT FOUND at {path}")

    print("\nFile-level breakdown (raw):")
    print(list_files_with_sizes(RAW_DIR).to_string(index=False))

    print("\nFile-level breakdown (working):")
    print(list_files_with_sizes(WORKING_DIR).to_string(index=False))

    # Quick row-count check on the working set
    print("\nLoading working set to report row/column counts...")
    working_files = [
        os.path.join(WORKING_DIR, f)
        for f in os.listdir(WORKING_DIR)
        if f.endswith(".parquet") or f.endswith(".csv")
    ]

    total_rows = 0
    for f in working_files:
        df = pd.read_parquet(f) if f.endswith(".parquet") else pd.read_csv(f)
        total_rows += len(df)
        print(f"  {os.path.basename(f)}: {len(df):,} rows, {df.shape[1]} columns")

    print(f"\nTotal rows in working set: {total_rows:,}")


DATASET SIZE REPORT (Raw & Working)
Raw          | NOT FOUND at data/raw
Working      |     5.97 GB | 132 file(s) | data/working

File-level breakdown (raw):
Empty DataFrame
Columns: [file, size_mb]
Index: []

File-level breakdown (working):
                           file  size_mb
yellow_tripdata_2013-01.parquet     0.00
yellow_tripdata_2013-02.parquet     0.00
yellow_tripdata_2013-03.parquet     0.00
yellow_tripdata_2013-04.parquet     0.00
yellow_tripdata_2013-05.parquet     0.00
yellow_tripdata_2013-06.parquet     0.00
yellow_tripdata_2013-07.parquet     0.00
yellow_tripdata_2013-08.parquet     0.00
yellow_tripdata_2013-09.parquet     0.00
yellow_tripdata_2013-10.parquet     0.00
yellow_tripdata_2013-11.parquet     0.00
yellow_tripdata_2013-12.parquet     0.00
yellow_tripdata_2014-01.parquet     0.00
yellow_tripdata_2014-02.parquet     0.00
yellow_tripdata_2014-03.parquet     0.00
yellow_tripdata_2014-04.parquet     0.00
yellow_tripdata_2014-05.parquet     0.00
yellow_tripdata_2014

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [29]:
import os
import shutil
import random

WORKING_DIR = "data/working"
PROCESSING_DIR = "data/processing"


def get_file_size_gb(path):
    return os.path.getsize(path) / (1024 ** 3)


def get_dir_size_gb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 ** 3)


def select_files_for_target_size(working_dir, target_gb=3, seed=42, shuffle=True):

    files = [
        os.path.join(working_dir, f)
        for f in os.listdir(working_dir)
        if f.endswith(".parquet") or f.endswith(".csv")
    ]
    files = sorted(files)  # deterministic starting order before optional shuffle

    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(files)

    selected = []
    running_total_gb = 0.0

    for f in files:
        size_gb = get_file_size_gb(f)
        selected.append((f, size_gb))
        running_total_gb += size_gb
        print(f"Added {os.path.basename(f)}: {size_gb:.2f} GB "
              f"(cumulative: {running_total_gb:.2f} GB)")
        if running_total_gb >= target_gb:
            break

    return selected, running_total_gb


if __name__ == "__main__":
    os.makedirs(PROCESSING_DIR, exist_ok=True)

    print("Selecting whole monthly files from the working set to build "
          "the ~3GB processing set (before cleaning/EDA)...")
    selected_files, total_size = select_files_for_target_size(
        WORKING_DIR, target_gb=3, seed=42, shuffle=True
    )

    print(f"\nCopying {len(selected_files)} file(s) into {PROCESSING_DIR}...")
    for f, size_gb in selected_files:
        dest = os.path.join(PROCESSING_DIR, os.path.basename(f))
        shutil.copy2(f, dest)

    actual_size = get_dir_size_gb(PROCESSING_DIR)
    print(f"\nProcessing set built: {actual_size:.2f} GB across {len(selected_files)} file(s)")
    print("Files included:", [os.path.basename(f) for f, _ in selected_files])


Selecting whole monthly files from the working set to build the ~3GB processing set (before cleaning/EDA)...
Added yellow_tripdata_2015-03.parquet: 0.00 GB (cumulative: 0.00 GB)
Added yellow_tripdata_2017-02.parquet: 0.12 GB (cumulative: 0.12 GB)
Added yellow_tripdata_2021-02.parquet: 0.02 GB (cumulative: 0.14 GB)
Added yellow_tripdata_2016-12.parquet: 0.00 GB (cumulative: 0.14 GB)
Added yellow_tripdata_2021-11.parquet: 0.05 GB (cumulative: 0.19 GB)
Added yellow_tripdata_2014-10.parquet: 0.00 GB (cumulative: 0.19 GB)
Added yellow_tripdata_2013-02.parquet: 0.00 GB (cumulative: 0.19 GB)
Added yellow_tripdata_2021-08.parquet: 0.04 GB (cumulative: 0.23 GB)
Added yellow_tripdata_2015-08.parquet: 0.00 GB (cumulative: 0.23 GB)
Added yellow_tripdata_2019-01.parquet: 0.10 GB (cumulative: 0.33 GB)
Added yellow_tripdata_2013-10.parquet: 0.00 GB (cumulative: 0.33 GB)
Added yellow_tripdata_2014-05.parquet: 0.00 GB (cumulative: 0.33 GB)
Added yellow_tripdata_2023-03.parquet: 0.05 GB (cumulative: 0.3

In [30]:

# data quality cleaning


import pandas as pd
import numpy as np
import glob
import os
import gc
import json

PROCESSING_DIR = "data/processing"
CLEANED_DIR = "data/processing/cleaned"
REMOVAL_LOG_PATH = "data/processing/cleaning_removal_log.json"

os.makedirs(CLEANED_DIR, exist_ok=True)

# Minimum reasonable size for a valid (even if empty) parquet file.
# A small parquet file will typically be at least a few hundred bytes.
MIN_PARQUET_FILE_SIZE_BYTES = 1024 # 1 KB

def get_processing_files(path=PROCESSING_DIR):
    all_files = glob.glob(f"{path}/*.parquet") + glob.glob(f"{path}/*.csv")
    valid_files = []
    for f in all_files:
        if "cleaned" not in f:
            if os.path.exists(f) and os.path.getsize(f) > MIN_PARQUET_FILE_SIZE_BYTES:
                valid_files.append(f)
            else:
                print(f"Skipping small or invalid file: {os.path.basename(f)} (size: {os.path.getsize(f)} bytes)")
    return sorted(valid_files)


def downcast_dtypes(df):
    """Reduce memory footprint by downcasting numeric columns to the
    smallest safe dtype."""
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    return df


def clean_taxi_data(df, pickup_col, dropoff_col, fare_col, distance_col, passenger_col):
    """
    Cleaning rules applied to a single file's worth of data.
    Returns cleaned df and a removal log dict for this file.
    """
    df = df.copy()
    n_before = len(df)
    log = {"start_rows": n_before}

    df[pickup_col] = pd.to_datetime(df[pickup_col], errors="coerce")
    df[dropoff_col] = pd.to_datetime(df[dropoff_col], errors="coerce")

    n = len(df)
    df = df.dropna(subset=[pickup_col, dropoff_col])
    log["unparseable_timestamps"] = n - len(df)

    df["trip_duration_min"] = (df[dropoff_col] - df[pickup_col]).dt.total_seconds() / 60
    n = len(df)
    df = df[(df["trip_duration_min"] > 0) & (df["trip_duration_min"] <= 240)]
    log["invalid_duration"] = n - len(df)

    n = len(df)
    df = df[df[fare_col] > 0]
    df = df[df[fare_col] < 200]  # fixed cap, consistent across files
    log["invalid_fare"] = n - len(df)

    n = len(df)
    df = df[(df[distance_col] > 0) & (df[distance_col] <= 100)]
    log["invalid_distance"] = n - len(df)

    n = len(df)
    df = df[(df[passenger_col] >= 1) & (df[passenger_col] <= 6)]
    log["invalid_passenger_count"] = n - len(df)

    n = len(df)
    df = df.drop_duplicates()
    log["duplicates"] = n - len(df)

    log["end_rows"] = len(df)
    log["total_removed"] = n_before - len(df)
    log["pct_removed"] = round(log["total_removed"] / n_before * 100, 2) if n_before else 0

    return df, log


def quality_report_single_file(df, filename):
    """Lightweight per-file quality snapshot."""
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    return {
        "file": filename,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_by_column": missing.to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
    }


if __name__ == "__main__":
    files = get_processing_files()
    print(f"Found {len(files)} file(s) in processing set")


    PICKUP_COL = "tpep_pickup_datetime"
    DROPOFF_COL = "tpep_dropoff_datetime"
    FARE_COL = "fare_amount"
    DISTANCE_COL = "trip_distance"
    PASSENGER_COL = "passenger_count"

    all_removal_logs = []
    per_file_quality = []

    for f in files:
        fname = os.path.basename(f)
        print(f"\nProcessing {fname}...")

        df = pd.read_parquet(f) if f.endswith(".parquet") else pd.read_csv(f)
        df = downcast_dtypes(df)

        per_file_quality.append(quality_report_single_file(df, fname))

        df_clean, removal_log = clean_taxi_data(
            df, PICKUP_COL, DROPOFF_COL, FARE_COL, DISTANCE_COL, PASSENGER_COL
        )
        removal_log["file"] = fname
        all_removal_logs.append(removal_log)

        out_path = os.path.join(CLEANED_DIR, f"cleaned_{fname.replace('.csv', '.parquet')}")
        df_clean.to_parquet(out_path, index=False)
        print(f"  {removal_log['start_rows']:,} -> {removal_log['end_rows']:,} rows "
              f"({removal_log['pct_removed']}% removed). Saved to {out_path}")

        del df, df_clean
        gc.collect()

    with open(REMOVAL_LOG_PATH, "w") as fp:
        json.dump({"per_file_removal": all_removal_logs}, fp, indent=2, default=str)

    with open("data/processing/per_file_quality_snapshot.json", "w") as fp:
        json.dump(per_file_quality, fp, indent=2, default=str)

    total_start = sum(l["start_rows"] for l in all_removal_logs)
    total_end = sum(l["end_rows"] for l in all_removal_logs)
    print("\n" + "=" * 50)
    print("AGGREGATE CLEANING SUMMARY (processing set)")
    print("=" * 50)
    print(f"Total rows before cleaning: {total_start:,}")
    print(f"Total rows after cleaning:  {total_end:,}")
    print(f"Total removed: {total_start - total_end:,} "
          f"({(total_start - total_end) / total_start * 100:.2f}%)")
    print(f"\nCleaned files saved individually in: {CLEANED_DIR}")

Skipping small or invalid file: yellow_tripdata_2016-05.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2016-06.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2013-02.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2016-01.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2013-10.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2016-12.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2015-07.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2016-07.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2013-08.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2016-03.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2014-11.parquet (size: 919 bytes)
Skipping small or invalid file: yellow_tripdata_2014-07.parquet (size: 919 bytes)
Skipping small o

In [ ]:
#EDA

# Install fastparquet for reading Parquet files with the specified engine
!pip install fastparquet

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")


def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved: {path}")
    plt.close(fig)


def summary_statistics(df, numeric_cols):
    stats = df[numeric_cols].describe().T
    stats["skew"] = df[numeric_cols].skew()
    stats["kurtosis"] = df[numeric_cols].kurtosis()
    print(stats)
    return stats


def plot_distribution(df, column, title, filename, bins=50, xlim=None):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df[column], bins=bins, ax=ax, kde=True)
    ax.set_title(title)
    ax.set_xlabel(column)
    if xlim:
        ax.set_xlim(xlim)
    save_fig(fig, filename)


def plot_trips_over_time(df, datetime_col, freq="D", title="Trips over time", filename="trips_over_time.png"):
    ts = df.set_index(datetime_col).resample(freq).size()
    fig, ax = plt.subplots(figsize=(10, 5))
    ts.plot(ax=ax)
    ax.set_title(title)
    ax.set_ylabel("Number of trips")
    save_fig(fig, filename)
    return ts


def plot_trips_by_hour_dow(df, datetime_col):
    tmp = df.copy()
    tmp["hour"] = tmp[datetime_col].dt.hour
    tmp["dow"] = tmp[datetime_col].dt.day_name()
    pivot = tmp.groupby(["dow", "hour"]).size().unstack(fill_value=0)

    dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    pivot = pivot.reindex(dow_order)

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(pivot, cmap="YlOrRd", ax=ax)
    ax.set_title("Trip volume by hour of day and day of week")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Day of week")
    save_fig(fig, "trips_by_hour_dow_heatmap.png")


def plot_monthly_volume(df, datetime_col):
    tmp = df.copy()
    tmp["year_month"] = tmp[datetime_col].dt.to_period("M")
    monthly = tmp.groupby("year_month").size()

    fig, ax = plt.subplots(figsize=(10, 5))
    monthly.plot(kind="bar", ax=ax)
    ax.set_title("Trip volume by month (processing set)")
    ax.set_ylabel("Number of trips")
    plt.xticks(rotation=45)
    save_fig(fig, "monthly_volume.png")
    return monthly


def plot_correlation_matrix(df, numeric_cols):
    fig, ax = plt.subplots(figsize=(8, 6))
    corr = df[numeric_cols].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlation matrix")
    save_fig(fig, "correlation_matrix.png")
    return corr


def plot_categorical_counts(df, column, title, filename, top_n=10):
    fig, ax = plt.subplots(figsize=(8, 5))
    df[column].value_counts().head(top_n).plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_ylabel("Count")
    save_fig(fig, filename)


if __name__ == "__main__":
    # Load the CLEANED PROCESSING dataset (small enough to load in one go)
    df = pd.read_parquet("data/processing/cleaned/", engine='fastparquet')


    DROPOFF_COL = "tpep_dropoff_datetime"
    PICKUP_COL = "tpep_pickup_datetime"
    FARE_COL = "fare_amount"
    DISTANCE_COL = "trip_distance"
    PASSENGER_COL = "passenger_count"
    NUMERIC_COLS = [FARE_COL, DISTANCE_COL, PASSENGER_COL, "trip_duration_min"]

    print("=" * 50)
    print("SUMMARY STATISTICS (Processing set)")
    print("=" * 50)
    summary_statistics(df, NUMERIC_COLS)

    print("\nGenerating distribution plots...")
    plot_distribution(df, FARE_COL, "Distribution of fare amount", "fare_distribution.png", xlim=(0, 100))
    plot_distribution(df, DISTANCE_COL, "Distribution of trip distance", "distance_distribution.png", xlim=(0, 30))
    plot_distribution(df, "trip_duration_min", "Distribution of trip duration (minutes)", "duration_distribution.png")

    print("\nGenerating time-series plot (evidence for Velocity)...")
    plot_trips_over_time(df, PICKUP_COL, freq="D", title="Daily trip volume")

    print("\nGenerating monthly volume plot...")
    monthly_counts = plot_monthly_volume(df, PICKUP_COL)
    print(monthly_counts)

    print("\nGenerating hour x day-of-week heatmap...")
    plot_trips_by_hour_dow(df, PICKUP_COL)

    print("\nGenerating correlation matrix...")
    plot_correlation_matrix(df, NUMERIC_COLS)

    print("\nGenerating passenger count distribution...")
    plot_categorical_counts(df, PASSENGER_COL, "Passenger count distribution", "passenger_count.png")

    print("\nAll figures saved to figures/., "
          "noting they describe the processing subset.")